# SNO+ (`SNOplus-like_opticks.gdml`): geometry and hits per PMT

Companion notebook of `GDML/SNOplus-like_opticks.gdml` (made by `scripts/make_snoplus_gdml.py`).
Reads the geometry straight from the GDML file and the `OpticksHits` tree written by OpticalSims with
**uproot**; no ROOT or Geant4 needed.

| Function | What it does |
|---|---|
| `read_gdml_sensors(gdml)` | every PMT placement (`SensDet = PhotonDetector`): copy number, aperture centre, inward direction |
| `snoplus_outline(gdml)` | AV, neck, scintillator level, PSUP and cavity dimensions read from the GDML solids |
| `draw_geometry_section(outline, sensors)` | x-z cross-section: cavity, PSUP, AV, neck, LS / N2, PMTs near the slice |
| `draw_sensor_map(sensors)` | equal-area (Mollweide) map of all PMTs, each drawn at its real angular size |
| `draw_sensors_3d(sensors, counts=None)` | 3D view, each PMT a disc of its real aperture, tangent to the PSUP |
| `load_hits`, `match_hits`, `hits_per_sensor` | `OpticksHits` -> DataFrame -> PMT index (SensorID = copy number, checked against x,y,z) |
| `plot_heatmap_map`, `plot_heatmap_3d` | hits per PMT on the flat map / in 3D |


## Configuration
Use **absolute paths** (`~` and `$VARS` are expanded). Relative paths are resolved from the directory
Jupyter was started in.

In [ ]:
from pathlib import Path

GDML_FILE = "OpticalSims/GDML/SNOplus-like_opticks.gdml"
ROOT_FILE = "opticks_test/test_SNOplus.root"   # gdml_det output
TREE_NAME = "OpticksHits"   # or "Geant4Hits"

EVENTS    = None            # None = all events, an int = one event, a list = several events
MATCH     = "id"            # "id": SensorID == copy number ; "position": nearest PMT to the hit x,y,z
LOG_COLOR = True
EVENT_DISPLAYS = 2          # how many single-event maps to draw

VERTEX    = (0.0, 0.0, 0.0) # mm, event vertex for the time residual (macros/SNOplus.mac fires from the centre)
N_LS, N_WATER = 1.50, 1.34  # effective (group) indices for the straight-line time of flight

SAVE_DIR  = "SNOplus"            # absolute path to also save full-quality PNGs
INLINE_FORMAT = "jpeg"      # "jpeg" (small), "png" (sharper, ~5x bigger) or None (nothing inline; use SAVE_DIR)
INLINE_DPI    = 100          # resolution of the embedded figures (notebook size grows ~ DPI^2)

In [ ]:
import os


def _find_by_name(name, roots, max_depth=5, limit=10):
    # Look for files called `name` under `roots` (bounded depth) to suggest the right path.
    hits = []
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if name in filenames:
                hits.append(Path(dirpath) / name)
                if len(hits) >= limit:
                    return hits
    return hits


def absolute_path(p, what="file", must_exist=True):
    # Expand ~ / $VARS, require an absolute path, and (optionally) check that it exists.
    p = Path(os.path.expandvars(os.path.expanduser(str(p))))
    if not p.is_absolute():
        print(f"WARNING: {what} path {str(p)!r} is relative; resolving it from {Path.cwd()}")
        p = Path.cwd() / p
    p = p.resolve()
    if must_exist and not p.exists():
        found = list(dict.fromkeys(_find_by_name(p.name, [Path.home(), p.parent.parent])))  # de-duplicated
        hint = ("\n  files with that name:\n    " + "\n    ".join(map(str, found))) if found else \
               f"\n  no file called {p.name!r} found under {Path.home()}"
        raise FileNotFoundError(f"{what} not found: {p}{hint}")
    return p


GDML_FILE = absolute_path(GDML_FILE, "GDML file")
ROOT_FILE = absolute_path(ROOT_FILE, "ROOT file")
SAVE_DIR = absolute_path(SAVE_DIR, "SAVE_DIR", must_exist=False) if SAVE_DIR else None
print("GDML_FILE =", GDML_FILE)
print("ROOT_FILE =", ROOT_FILE)
if SAVE_DIR:
    print("SAVE_DIR  =", SAVE_DIR)

## Imports

In [ ]:
import math
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.collections import PatchCollection, PolyCollection
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import Circle, Rectangle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import cKDTree
import uproot
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

plt.rcParams["figure.dpi"] = INLINE_DPI
if INLINE_FORMAT == "jpeg":
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 70, "optimize": True})
elif INLINE_FORMAT == "png":
    set_matplotlib_formats("png")
else:
    plt.ioff()
    set_matplotlib_formats("png")
print("uproot", uproot.__version__, "| numpy", np.__version__, "| pandas", pd.__version__,
      f"| inline figures: {INLINE_FORMAT or 'off'} @ {INLINE_DPI} dpi")


def _save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(SAVE_DIR) / f"{name}.png", dpi=150, bbox_inches="tight")
    if not INLINE_FORMAT:
        plt.close(fig)

## GDML reader
Same reader as `OpticksHits_heatmap.ipynb`: walks the volume tree with Geant4's placement convention
(`M = Rz·Ry·Rx`, placement uses `M⁻¹`). For `volPMT` the sensor origin is the centre of the concentrator
aperture and the local +x axis (`dir_*`) points into the detector.

In [ ]:
_UNITS = {"mm": 1.0, "cm": 10.0, "m": 1000.0, "km": 1e6, "um": 1e-3, "nm": 1e-6,
          "rad": 1.0, "mrad": 1e-3, "deg": math.pi / 180.0, "degree": math.pi / 180.0}
_ANG_DEFAULT, _LEN_DEFAULT = "rad", "mm"     # GDML defaults


class _Expr:
    # Evaluates GDML numeric attributes ("16900", "2*R", "180*deg", "pi/2") with <define> constants.
    def __init__(self):
        self.ns = {"pi": math.pi, "e": math.e, "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
                   "tan": math.tan, "asin": math.asin, "acos": math.acos, "atan": math.atan,
                   "atan2": math.atan2, "exp": math.exp, "log": math.log, "abs": abs, "pow": pow}
        self.ns.update(_UNITS)

    def __call__(self, s, default=0.0):
        if s is None:
            return default
        try:
            return float(s)
        except ValueError:
            return float(eval(str(s).replace("^", "**"), {"__builtins__": {}}, self.ns))

    def define(self, name, value):
        self.ns[name] = self(value)


def _rot_matrix(ax, ay, az):
    # CLHEP: rot.rotateX(ax); rot.rotateY(ay); rot.rotateZ(az)  ->  M = Rz @ Ry @ Rx
    cx, sx, cy, sy, cz, sz = math.cos(ax), math.sin(ax), math.cos(ay), math.sin(ay), math.cos(az), math.sin(az)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def read_gdml_sensors(gdml_path, sensor_aux=("SensDet", "PD"), sensor_value="PhotonDetector",
                      sensor_lvs=None, verbose=True):
    # Return a DataFrame with one row per sensitive placement:
    #   copynumber, lv, pv, x, y, z [mm, sensor-volume origin], dir_x/y/z (sensor local +x in global frame),
    #   depth, surface ('barrel' / 'top' / 'bottom', for cylinder-like detectors)
    # Sensors = LVs carrying <auxiliary auxtype in sensor_aux auxvalue=sensor_value/> (as OpticalSims'
    # DetectorConstruction does), or the explicit list `sensor_lvs`.
    root = ET.parse(str(gdml_path)).getroot()
    for el in root.iter():                          # drop any XML namespace prefixes
        if isinstance(el.tag, str) and "}" in el.tag:
            el.tag = el.tag.split("}", 1)[1]
    ev = _Expr()

    positions, rotations = {}, {}
    for d in root.findall("define"):
        for el in d:
            name = el.get("name")
            if el.tag in ("constant", "variable"):
                ev.define(name, el.get("value"))
            elif el.tag == "quantity":
                ev.define(name, f"({el.get('value')})*{el.get('unit', '1')}")
    for d in root.findall("define"):
        for el in d:
            if el.tag == "position":
                positions[el.get("name")] = el
            elif el.tag == "rotation":
                rotations[el.get("name")] = el

    def vec(el, kind):
        if el is None:
            return np.zeros(3)
        u = _UNITS.get(el.get("unit", _LEN_DEFAULT if kind == "pos" else _ANG_DEFAULT), 1.0)
        return np.array([ev(el.get(k), 0.0) for k in "xyz"]) * u

    structure = root.find("structure")
    volumes = {v.get("name"): v for v in structure if v.tag in ("volume", "assembly")}

    sensors = set(sensor_lvs or [])
    if not sensor_lvs:
        for name, v in volumes.items():
            for aux in v.findall("auxiliary"):
                if aux.get("auxtype") in sensor_aux and aux.get("auxvalue") == sensor_value:
                    sensors.add(name)
    if not sensors:
        raise RuntimeError("no sensitive volumes found (no SensDet/PD = PhotonDetector auxiliary)")

    world = root.find("setup/world").get("ref")
    rows, stack = [], [(world, np.zeros(3), np.eye(3), 0)]
    while stack:
        lv, t, R, depth = stack.pop()
        for pv in volumes[lv].findall("physvol"):
            ref = pv.find("volumeref")
            if ref is None:
                raise NotImplementedError("physvol without volumeref (inline <file>) is not supported")
            child = ref.get("ref")
            pos = pv.find("position")
            if pos is None and pv.find("positionref") is not None:
                pos = positions[pv.find("positionref").get("ref")]
            rot = pv.find("rotation")
            if rot is None and pv.find("rotationref") is not None:
                rot = rotations[pv.find("rotationref").get("ref")]
            Rloc = _rot_matrix(*vec(rot, "rot")).T          # GDML -> Geant4 uses the inverse
            tg, Rg = t + R @ vec(pos, "pos"), R @ Rloc
            if child in sensors:
                d = Rg[:, 0]
                rows.append((int(pv.get("copynumber", 0)), child, pv.get("name", child + "_PV"),
                             *tg, *d, depth + 1))
            elif child in volumes:
                stack.append((child, tg, Rg, depth + 1))

    df = pd.DataFrame(rows, columns=["copynumber", "lv", "pv", "x", "y", "z",
                                     "dir_x", "dir_y", "dir_z", "depth"])
    df = df.sort_values("copynumber", kind="stable").reset_index(drop=True)

    # cylinder bookkeeping: caps are the sensors sitting at the extreme z planes
    tol = 1.0  # mm
    zmax, zmin = df.z.max(), df.z.min()
    df["surface"] = np.where(df.z > zmax - tol, "top", np.where(df.z < zmin + tol, "bottom", "barrel"))
    df.attrs["sensor_radius"] = _sensor_radius(root, volumes, sorted(sensors), ev)
    df.attrs["gdml"] = str(gdml_path)

    if verbose:
        print(f"{gdml_path}: {len(df)} sensors in LVs {sorted(sensors)}")
        print(df.groupby(["lv", "surface"]).size().rename("n").to_string())
        dup = df.copynumber.duplicated().sum()
        if dup:
            print(f"WARNING: {dup} duplicated copy numbers -> SensorID is not unique per sensor")
        print(f"sensor radius ~ {df.attrs['sensor_radius']} mm")
    return df


def _sensor_radius(root, volumes, sensor_lvs, ev):
    # Radius of the first <orb>/<sphere> found in a sensor LV's solid tree (None if there is none).
    solids = {s.get("name"): s for s in root.find("solids")}
    for lv in sensor_lvs:
        todo = [volumes[lv].find("solidref").get("ref")]
        while todo:
            s = solids.get(todo.pop())
            if s is None:
                continue
            u = _UNITS.get(s.get("lunit", "mm"), 1.0)
            if s.tag == "orb":
                return ev(s.get("r")) * u
            if s.tag == "sphere":
                return ev(s.get("rmax")) * u
            todo += [c.get("ref") for c in s if c.tag in ("first", "second")]
    return None

## SNO+ outline and sensor table

In [ ]:
def snoplus_outline(gdml_path):
    # Dimensions (mm) of the main volumes, read from the solids written by make_snoplus_gdml.py.
    root = ET.parse(str(gdml_path)).getroot()
    ev = _Expr()
    solids = {s.get("name"): s for s in root.find("solids")}

    def num(solid, key):
        s = solids.get(solid)
        if s is None:
            return None
        return ev(s.get(key)) * _UNITS.get(s.get("lunit", "mm"), 1.0)

    def second_z(union):
        s = solids.get(union)
        pos = s.find("position") if s is not None else None
        return ev(pos.get("z")) * _UNITS.get(pos.get("unit", "mm"), 1.0) if pos is not None else 0.0

    av_place = read_gdml_sensors(gdml_path, sensor_lvs=["volAV"], verbose=False)
    o = dict(av_centre=av_place[["x", "y", "z"]].to_numpy()[0] if len(av_place) else np.zeros(3))
    o["av_rin"], o["av_rout"] = num("ScintSphere", "r"), num("AVSphere", "r")
    o["psup_rin"], o["psup_rout"] = num("PSUPInner", "r"), num("PSUPOuter", "r")
    o["psup_hole_r"] = num("PSUPHoleTube", "rmax")
    for key, solid, union in (("neck_out", "AVNeckOuter", "AVSolid"), ("ls_neck", "ScintNeck", "ScintSolid")):
        r, h, zc = num(solid, "rmax"), num(solid, "z"), second_z(union)
        o[key] = (r, zc - h / 2, zc + h / 2) if r else None
    gas = solids.get("NeckGasSolid")
    if gas is not None:
        g = read_gdml_sensors(gdml_path, sensor_lvs=["volNeckGas"], verbose=False)
        zc = float(g.z.iloc[0]) if len(g) else 0.0
        h = num("NeckGasSolid", "z")
        o["neck_gas"] = (num("NeckGasSolid", "rmax"), zc - h / 2, zc + h / 2)
    cav = read_gdml_sensors(gdml_path, sensor_lvs=["volCavity"], verbose=False)
    if len(cav) and num("CavityTube", "rmax"):
        zc, h = float(cav.z.iloc[0]), num("CavityTube", "z")
        o["cavity"] = (num("CavityTube", "rmax"), zc - h / 2, zc + h / 2)
    print("outline [mm]:", {k: (np.round(v, 1) if v is not None else None) for k, v in o.items()})
    return o


def add_angles(sensors, centre=(0.0, 0.0, 0.0)):
    # theta/phi of every PMT seen from `centre` (AV centre) + its aperture radius.
    p = sensors[["x", "y", "z"]].to_numpy() - np.asarray(centre)
    r = np.linalg.norm(p, axis=1)
    sensors["r"] = r
    sensors["theta"] = np.arccos(np.clip(p[:, 2] / r, -1, 1))
    sensors["phi"] = np.arctan2(p[:, 1], p[:, 0])
    return sensors

## Geometry plots

In [ ]:
def draw_geometry_section(outline, sensors, slab=400.0, unit="m"):
    # x-z cross-section through the AV centre: volumes as outlines, PMTs with |y| < slab mm as their aperture.
    s = {"m": 1e-3, "cm": 0.1, "mm": 1.0}[unit]
    cx, _, cz = outline["av_centre"]
    fig, ax = plt.subplots(figsize=(9, 11))
    if outline.get("cavity"):
        R, z0, z1 = outline["cavity"]
        ax.add_patch(Rectangle((-R * s, z0 * s), 2 * R * s, (z1 - z0) * s, fc="#dbe9f6", ec="0.4", lw=1, label="cavity water"))
    for r0, r1, fc, lab in ((outline["psup_rin"], outline["psup_rout"], "0.25", "PSUP"),
                            (outline["av_rin"], outline["av_rout"], "#c7a96b", "acrylic vessel")):
        if r0 and r1:
            ax.add_patch(Circle((cx * s, cz * s), r1 * s, fc=fc, ec="none", label=lab))
            ax.add_patch(Circle((cx * s, cz * s), r0 * s, fc="#dbe9f6" if lab == "PSUP" else "#f2d16b", ec="none"))
    if outline.get("psup_hole_r"):
        h = outline["psup_hole_r"]
        ax.add_patch(Rectangle(((cx - h) * s, (cz + outline["psup_rin"] - 50) * s), 2 * h * s,
                               (outline["psup_rout"] - outline["psup_rin"] + 100) * s, fc="#dbe9f6", ec="none"))
    if outline.get("neck_out"):
        r, z0, z1 = outline["neck_out"]
        ax.add_patch(Rectangle(((cx - r) * s, (cz + z0) * s), 2 * r * s, (z1 - z0) * s, fc="#c7a96b", ec="none"))
    if outline.get("ls_neck"):
        r, z0, z1 = outline["ls_neck"]
        ax.add_patch(Rectangle(((cx - r) * s, (cz + z0) * s), 2 * r * s, (z1 - z0) * s, fc="#f2d16b", ec="none",
                               label="LAB + PPO"))
    if outline.get("neck_gas"):
        r, z0, z1 = outline["neck_gas"]
        ax.add_patch(Rectangle(((cx - r) * s, z0 * s), 2 * r * s, (z1 - z0) * s, fc="white", ec="0.6", lw=0.5,
                               label="N$_2$ cover gas"))
    near = sensors[np.abs(sensors.y - cx) < slab]
    rs = sensors.attrs.get("sensor_radius") or 130.0
    for _, p in near.iterrows():                     # aperture as a segment perpendicular to the inward axis
        d = np.array([p.dir_x, p.dir_z], float)
        d = d / (np.linalg.norm(d) or 1.0)
        t = np.array([-d[1], d[0]]) * rs
        ax.plot([(p.x - t[0]) * s, (p.x + t[0]) * s], [(p.z - t[1]) * s, (p.z + t[1]) * s], color="C3", lw=2)
    ax.plot([], [], color="C3", lw=2, label=f"PMT apertures (|y| < {slab:.0f} mm)")
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.set(xlabel=f"x [{unit}]", ylabel=f"z [{unit}]", title="SNO+ x-z section")
    ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    _save(fig, "geometry_section")
    return fig


def mollweide(lon, lat, iters=12):
    # Equal-area Mollweide projection (radians in, unit-sphere map coordinates out).
    lat = np.asarray(lat, float)
    th = lat.copy()
    target = np.pi * np.sin(lat)
    for _ in range(iters):                       # Newton: 2t + sin 2t = pi sin(lat)
        f = 2 * th + np.sin(2 * th) - target
        th -= f / np.where(np.abs(np.cos(th)) < 1e-9, 1e-9, 2 + 2 * np.cos(2 * th))
    return 2 * np.sqrt(2) / np.pi * np.asarray(lon) * np.cos(th), np.sqrt(2) * np.sin(th)


def _map_polys(sensors, npts=12):
    # Every PMT aperture projected onto the map: its outline on the sphere (angular radius aperture/r).
    rs = sensors.attrs.get("sensor_radius") or 130.0
    n = sensors[["x", "y", "z"]].to_numpy()
    n = n / np.linalg.norm(n, axis=1)[:, None]
    a = np.cross(n, np.where(np.abs(n[:, 2:3]) < 0.9, [[0, 0, 1]], [[1, 0, 0]]))
    a /= np.linalg.norm(a, axis=1)[:, None]
    b = np.cross(n, a)
    al = (rs / sensors.r.to_numpy())[:, None, None]
    psi = np.linspace(0, 2 * np.pi, npts, endpoint=False)[None, :, None]
    q = n[:, None, :] * np.cos(al) + (a[:, None, :] * np.cos(psi) + b[:, None, :] * np.sin(psi)) * np.sin(al)
    lon = np.arctan2(q[..., 1], q[..., 0])
    lat = np.arcsin(np.clip(q[..., 2], -1, 1))
    lon0 = sensors.phi.to_numpy()[:, None]
    lon = lon0 + (lon - lon0 + np.pi) % (2 * np.pi) - np.pi   # keep each outline continuous at lon = +-pi
    x, y = mollweide(lon, lat)
    return np.stack([x, y], axis=-1)


def _map_frame(ax):
    # Map outline, graticule and labels; returns the outline patch (used as clip path).
    t = np.linspace(-np.pi / 2, np.pi / 2, 200)
    ex, ey = mollweide(np.full_like(t, np.pi), t)
    outline = ax.fill(np.r_[ex, -ex[::-1]], np.r_[ey, ey[::-1]], fc="none", ec="0.3", lw=0.8)[0]
    for lo in np.radians(np.arange(-150, 181, 30)):
        x, y = mollweide(np.full_like(t, lo), t)
        ax.plot(x, y, color="0.75", lw=0.4, zorder=0)
    for la in np.radians(np.arange(-60, 61, 30)):
        x, y = mollweide(np.linspace(-np.pi, np.pi, 100), np.full(100, la))
        ax.plot(x, y, color="0.75", lw=0.4, zorder=0)
        ax.annotate(f"{90 - np.degrees(la):.0f}°", (x[0], y[0]), xytext=(-24, -3), textcoords="offset points", fontsize=7)
    for lo in (-120, -60, 0, 60, 120):
        x, _ = mollweide(np.radians(lo), 0.0)
        ax.annotate(f"{lo}°", (x, 0), xytext=(2, 2), textcoords="offset points", fontsize=7, color="0.4")
    ax.set_aspect("equal")
    ax.set_xlim(-2.9, 2.9)
    ax.set_ylim(-1.5, 1.5)
    ax.axis("off")
    ax.text(0, 1.47, r"$\theta$ = 0 (top, neck)", ha="center", fontsize=8)
    ax.text(0, -1.53, r"$\theta$ = 180°", ha="center", va="top", fontsize=8)
    return outline


def draw_sensor_map(sensors, color="C0"):
    # Equal-area (Mollweide) map of every PMT: longitude = phi, latitude = 90 deg - theta.
    fig, ax = plt.subplots(figsize=(12, 6.5))
    frame = _map_frame(ax)
    pc = PolyCollection(_map_polys(sensors), facecolor=color, edgecolor="none")
    ax.add_collection(pc)
    pc.set_clip_path(frame)
    ax.set_title(f"{len(sensors)} PMTs - {Path(sensors.attrs.get('gdml', '')).name} (Mollweide)")
    fig.tight_layout()
    _save(fig, "sensor_map")
    return fig


def _disc_polys(sensors, s, npts=10):
    # Each PMT as a polygon of its aperture radius in the plane perpendicular to its inward axis.
    rs = sensors.attrs.get("sensor_radius") or 130.0
    c = sensors[["x", "y", "z"]].to_numpy() * s
    d = sensors[["dir_x", "dir_y", "dir_z"]].to_numpy(dtype=float, copy=True)
    d = d / np.linalg.norm(d, axis=1)[:, None]
    a = np.cross(d, np.where(np.abs(d[:, 2:3]) < 0.9, [[0, 0, 1]], [[1, 0, 0]]))
    a /= np.linalg.norm(a, axis=1)[:, None]
    b = np.cross(d, a)
    ang = np.linspace(0, 2 * np.pi, npts, endpoint=False)
    return c[:, None, :] + rs * s * (np.cos(ang)[None, :, None] * a[:, None, :] + np.sin(ang)[None, :, None] * b[:, None, :])


def _norm(counts, log):
    # Log colour scale for wide ranges; linear (integer-friendly) when the counts span < 20x.
    hit = counts > 0
    vmax = counts.max() if hit.any() else 1
    vmin = counts[hit].min() if hit.any() else 1
    if log and vmax / max(vmin, 1e-9) >= 20:
        return LogNorm(vmin=max(vmin, 1e-3), vmax=vmax)
    return Normalize(vmin=min(vmin, 1) if hit.any() else 0, vmax=max(vmax, 1))


def draw_sensors_3d(sensors, counts=None, log=LOG_COLOR, cmap="viridis", zero_color="0.82", elev=20, azim=-60,
                    hemisphere=None, title=None, unit="m", save_as="sensors_3d"):
    # 3D view: PMTs as discs tangent to the PSUP; coloured by `counts` if given.
    #   hemisphere="far" draws only the far half of the sphere (look inside it), "near" the half facing you.
    s = {"m": 1e-3, "cm": 0.1, "mm": 1.0}[unit]
    polys = _disc_polys(sensors, s)
    keep = np.ones(len(sensors), bool)
    if hemisphere:
        el, az = np.radians(elev), np.radians(azim)
        view = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
        dot = sensors[["x", "y", "z"]].to_numpy() @ view
        keep = dot < 0 if hemisphere == "far" else dot > 0
    fig = plt.figure(figsize=(10, 9))
    ax = fig.add_subplot(projection="3d")
    if counts is None:
        ax.add_collection3d(Poly3DCollection(polys[keep], facecolor="C0", edgecolor="none", alpha=0.9))
    else:
        counts = np.asarray(counts, float)
        norm, cm = _norm(counts, log), plt.get_cmap(cmap)
        hit = (counts > 0) & keep
        ax.add_collection3d(Poly3DCollection(polys[~hit & keep], facecolor=zero_color, edgecolor="none", alpha=0.35))
        ax.add_collection3d(Poly3DCollection(polys[hit], facecolors=cm(norm(counts[hit])), edgecolor="none"))
        fig.colorbar(ScalarMappable(norm=norm, cmap=cm), ax=ax, shrink=0.6, pad=0.1, label="hits")
    R = sensors.r.max() * s * 1.05
    ax.set(xlim=(-R, R), ylim=(-R, R), zlim=(-R, R), xlabel=f"x [{unit}]", ylabel=f"y [{unit}]", zlabel=f"z [{unit}]")
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title or f"{int(keep.sum())} PMTs")
    _save(fig, save_as)
    return fig

## Hits

In [ ]:
HIT_BRANCHES = ["evtID", "hit_Id", "SensorID", "x", "y", "z", "t", "wavelength", "boundary"]


def load_hits(root_file, tree=TREE_NAME, events=None):
    with uproot.open(root_file) as f:
        if tree not in f:
            raise KeyError(f"{tree!r} not in {root_file}; keys: {f.keys()}")
        t = f[tree]
        df = t.arrays([b for b in HIT_BRANCHES if b in t.keys()], library="pd")
    if events is not None:
        ev = [events] if np.isscalar(events) else list(events)
        df = df[df.evtID.isin(ev)]
    df = df.reset_index(drop=True)
    print(f"{Path(root_file).name}:{tree}: {len(df):,} hits in {df.evtID.nunique()} event(s)")
    return df


def match_hits(hits, sensors, how=MATCH):
    # Adds 'pmt' (row in `sensors`, -1 = unknown) and 'pmt_dist' (hit to aperture centre, mm).
    tree = cKDTree(sensors[["x", "y", "z"]].to_numpy())
    dist, near = tree.query(hits[["x", "y", "z"]].to_numpy(float))
    if how == "id":
        lut = pd.Series(np.arange(len(sensors)), index=sensors.copynumber.to_numpy())
        hits["pmt"] = hits.SensorID.map(lut).fillna(-1).astype(int)
        agree = (hits.pmt.to_numpy() == near).mean()
        print(f"SensorID -> PMT: {(hits.pmt < 0).sum()} unknown ids; nearest PMT agrees for {agree:.2%} of hits")
    else:
        hits["pmt"] = near
    ok = hits.pmt.to_numpy() >= 0
    p = sensors[["x", "y", "z"]].to_numpy()[np.where(ok, hits.pmt, 0)]
    hits["pmt_dist"] = np.where(ok, np.linalg.norm(hits[["x", "y", "z"]].to_numpy(float) - p, axis=1), np.nan)
    return hits


def hits_per_sensor(hits, sensors):
    sel = hits.pmt >= 0
    return np.bincount(hits.pmt[sel], minlength=len(sensors)).astype(float)


def time_residual(hits, vertex=VERTEX, n_ls=N_LS, n_water=N_WATER, r_av=None):
    # t - straight-line time of flight from `vertex`, split into the LS (r < r_av) and water parts.
    c = 299.792458  # mm/ns
    v = np.asarray(vertex, float)
    P = hits[["x", "y", "z"]].to_numpy(float)
    d = P - v
    L = np.linalg.norm(d, axis=1)
    u = d / L[:, None]
    r_av = r_av or OUTLINE.get("av_rin") or 6005.0
    b = u @ v
    disc = b ** 2 - (v @ v - r_av ** 2)
    l_ls = np.clip(-b + np.sqrt(np.clip(disc, 0, None)), 0, L)    # path until leaving the AV sphere
    tof = (l_ls * n_ls + (L - l_ls) * n_water) / c
    return hits.t.to_numpy() - tof

## Heat maps and checks

In [ ]:
def plot_heatmap_map(sensors, counts, log=LOG_COLOR, cmap="viridis", zero_color="0.85", title="hits per PMT",
                     save_as="heatmap_map"):
    # Hits per PMT on the equal-area map; every PMT drawn at its real angular size.
    counts = np.asarray(counts, float)
    hit = counts > 0
    polys = _map_polys(sensors)
    fig, ax = plt.subplots(figsize=(12, 7))
    frame = _map_frame(ax)
    blank = PolyCollection(polys[~hit], facecolor=zero_color, edgecolor="none")
    order = np.argsort(counts[hit])
    lit = PolyCollection(polys[hit][order], cmap=cmap, norm=_norm(counts, log), edgecolor="none")
    lit.set_array(counts[hit][order])
    for pc in (blank, lit):
        ax.add_collection(pc)
        pc.set_clip_path(frame)
    fig.colorbar(lit, ax=ax, orientation="horizontal", shrink=0.6, pad=0.06, label="hits")
    ax.set_title(f"{title}: {counts.sum():,.0f} hits on {hit.sum()}/{len(counts)} PMTs")
    _save(fig, save_as)
    return fig


def plot_hit_checks(hits, sensors, bins=100, save_as="hit_checks"):
    fig, axs = plt.subplots(2, 3, figsize=(16, 8))
    axs = axs.ravel()
    nh = hits.groupby("evtID").size()
    axs[0].hist(nh, bins=min(50, max(5, len(nh))))
    axs[0].set(xlabel="hits per event", ylabel="events", title=f"Nhits: mean {nh.mean():,.0f}")
    axs[1].hist(hits.t, bins=bins, range=(hits.t.min(), np.percentile(hits.t, 99.5)))
    axs[1].set(xlabel="t [ns]", ylabel="hits", yscale="log", title="hit time")
    tres = time_residual(hits)
    t0 = np.percentile(tres, 0.5)
    axs[2].hist(tres, bins=bins, range=(t0 - 10, t0 + 120))
    axs[2].set(xlabel="t - ToF [ns]", ylabel="hits", yscale="log", title=f"time residual, vertex {VERTEX} mm")
    if "wavelength" in hits:
        axs[3].hist(hits.wavelength, bins=bins)
        axs[3].set(xlabel="wavelength [nm]", ylabel="hits", title="hit wavelength")
    axs[4].hist(hits.pmt_dist, bins=bins, range=(0, 1.5 * (sensors.attrs.get("sensor_radius") or 130.0)))
    axs[4].axvline(sensors.attrs.get("sensor_radius") or 130.0, color="C3", ls="--", label="aperture radius")
    axs[4].set(xlabel="|hit - aperture centre| [mm]", ylabel="hits", title="hit on its PMT dome")
    axs[4].legend()
    ct = np.cos(sensors.theta.to_numpy()[hits.pmt[hits.pmt >= 0]])
    axs[5].hist(ct, bins=40, range=(-1, 1))
    axs[5].set(xlabel=r"cos$\theta_{PMT}$", ylabel="hits", title="hits vs PMT polar angle")
    fig.tight_layout()
    _save(fig, save_as)
    return fig

## 1. Geometry

In [ ]:
sensors = add_angles(read_gdml_sensors(GDML_FILE, verbose=False).drop(columns="surface"))
print(f"{GDML_FILE.name}: {len(sensors)} PMTs in LV(s) {sorted(sensors.lv.unique())}, copy numbers "
      f"{sensors.copynumber.min()}..{sensors.copynumber.max()} (unique: {sensors.copynumber.is_unique}), "
      f"aperture radius {sensors.attrs.get('sensor_radius')} mm, at r = {sensors.r.min():.0f}..{sensors.r.max():.0f} mm")
OUTLINE = snoplus_outline(GDML_FILE)
sensors.head()

In [ ]:
draw_geometry_section(OUTLINE, sensors);

In [ ]:
draw_sensor_map(sensors);

In [ ]:
draw_sensors_3d(sensors, title=f"{len(sensors)} PMTs (3D)");

## 2. Hits

In [ ]:
hits = load_hits(ROOT_FILE, events=EVENTS)
hits = match_hits(hits, sensors, how=MATCH)
counts = hits_per_sensor(hits, sensors)
hits.head()

## 3. Heat maps - all selected events

In [ ]:
plot_heatmap_map(sensors, counts, title=ROOT_FILE.name);

In [ ]:
draw_sensors_3d(sensors, counts, title="hits per PMT (3D)", save_as="heatmap_3d");

## 4. One event at a time

In [ ]:
for evt in sorted(hits.evtID.unique())[:EVENT_DISPLAYS]:
    c = hits_per_sensor(hits[hits.evtID == evt], sensors)
    plot_heatmap_map(sensors, c, title=f"event {evt}", save_as=f"heatmap_map_evt{evt}")

In [ ]:
plot_hit_checks(hits, sensors);

In [ ]:
sensors.assign(hits=counts).nlargest(10, "hits")[["copynumber", "x", "y", "z", "theta", "phi", "hits"]]